# 2. User Log Data Generation

The purpose of this notebook is to take the users in the cohorts of the mult_cohort_transaction_data.csv and calculate usage velocities and number of days from last login. The user_log.csv is very large so we have to use polars to calculate the desired features from the file.

## 1. Setup and File Loading

We load in the mult_cohort_transaction_data.csv to get the cohort information. The cohort information we need are what cohorts we have and the users involved.

In [ ]:
import numpy as np
import pandas as pd

import os
import polars as pl
from datetime import datetime, timedelta

#Set the max threads to 1 to prevent my local machine from running out of ram during operation.
os.environ['POLARS_MAX_THREADS'] = '1'
print(pl.thread_pool_size())

1


In [ ]:
DATA_DIR = "../data/"

In [ ]:
trans = pd.read_csv(
    os.path.join(DATA_DIR, "processed/", "mult_cohort_transaction_data.csv"), parse_dates = ['cohort_cutoff_date']
)



In [ ]:
users = trans[['msno', 'cohort_cutoff_date']].copy()

In [ ]:
del trans

## 2. Polars Lazy Frame User Log Cohort Generator Setup
We set up the scheme for polars to generate the data that we want.

In [ ]:
#Initialize Lazy Scan of user logs
lf = pl.scan_csv(
    os.path.join(DATA_DIR, "raw/", "user_logs.csv"),
    schema_overrides={"date": pl.String}
).with_columns(
    pl.col("date").str.to_date(format="%Y%m%d")
)

#All features besides msno in user_log.csv
features = ['num_25', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']

#Custom function to setup polar scheme to generate user_log velocity data for the cohort (month, year)
#For efficiency, we pass in the users for the specified cohort as a lazy frame users_lazy
def process_cohort_user_log(month, year, users_lazy):
    history_cutoff = datetime(year, month, 1).date()

    #Find 14 days ago and 30 days ago from the cutoff
    start_14 = history_cutoff - timedelta(days=14)
    start_30 = history_cutoff - timedelta(days=30)


    # Filter raw logs and join to users_lazy, the users we care about for this cohort
    #We here restrict our attention to only usage before cut_off date
    lf_cohort = users_lazy.join(
        lf.filter(pl.col("date") < history_cutoff),
        on="msno",
        how="left"
    )

    # Build a single list of expressions to process all metrics simultaneously
    exprs = [
        #this expression calculates the days since last use for each user in user_lazy relative to the current cohort
        ((pl.lit(history_cutoff) - pl.col("date").max()).dt.total_days() - 1).alias("days_since_last_use")
    ]

    #this loop builds an expression to calculate the velocity of each usage feature
    for f in features:
        sum_30 = pl.col(f).filter(pl.col("date") >= start_30).fill_null(0).sum()
        sum_14 = pl.col(f).filter(pl.col("date") >= start_14).fill_null(0).sum()

        velocity_expr = (
            pl.when(sum_30 > 0)
            .then(sum_14 / sum_30)
            .otherwise(pl.lit(None))
            .alias(f"{f}_velocity")
        )
        exprs.append(velocity_expr)

    #Aggregate and run each expression in one pass.
    metrics_lf = lf_cohort.group_by(["msno", "cohort_cutoff_date"]).agg(exprs)

    return metrics_lf

## 3. Implementation

1.   We loop through to build a cohort plan for each cohort in question.
2.   We loop over each cohort plan actually evaluating and outputting to a parquet file.
3. We concat all the resultant parquet files into one parquet file.

In [ ]:
cohort_plans = []
for year in range(2015, 2018):
    for month in range(1, 13):
        # Skip condition, we don't calculate for the first month we have since
        #this is too close to the beginning
        if year == 2015 and month == 1:
            continue

        # Break condition, we break here as this is the end of our scope
        if year == 2017 and month == 3:
            break

        #Finds the relevant users from this cohort in the users data frame
        pandas_cohort_slice = users[users['cohort_cutoff_date'] == datetime(year, month, 1)]
        if pandas_cohort_slice.empty:
            print("no user churn for", month, year)
            continue

        #We cast the cutoff_date as a Date type and get a lazy frame version
        current_cohort_users = pl.from_pandas(pandas_cohort_slice).with_columns(
            pl.col("cohort_cutoff_date").cast(pl.Date)
        ).lazy()

        #Adds the current cohort plan to the cohorts_plan list
        cohort_plans.append(process_cohort_user_log(month, year, current_cohort_users))

In [ ]:
#Delete users to save ram
del users


In [ ]:
#We generate one parquet file per cohort and output the results to a temp file
file_num = 0
for plan in cohort_plans:
    file_num += 1
    print("working on", file_num)
    output_file = os.path.join(DATA_DIR, "processed/", f"mult_cohort_usage_data_{file_num}.parquet")
    plan.sink_parquet(output_file, engine="streaming")


working on 23
working on 24
working on 25


In [ ]:
#We read in each separate parquet file and combine into a single parquet file
df = pd.DataFrame()

for file_num in range(1,26):
    input_file = os.path.join(DATA_DIR, "processed/", f"mult_cohort_usage_data_{file_num}.parquet")
    temp = pd.read_parquet(input_file)
    os.remove(input_file)
    df = pd.concat([df, temp])


In [ ]:
output_file = os.path.join(DATA_DIR, "processed/", "mult_cohort_usage_data.parquet")
df.to_parquet(output_file)